# Imports

In [4]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

ModuleNotFoundError: No module named 'cyvcf2'

# Load Data

In [2]:
vcf = VCF("data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("data/ps3_gwas.phen", sep="\t", header=None)

# Clean and Align Phenotype Data

In [4]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS Loop

In [ ]:
# Set up output file
with open("gwas_results.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    max_snps = 50
    count = 0

    for v in vcf:
        # if count >= max_snps:
        #     break

        # Convert genotype to dosage-like numeric (0/1/2); missing -> nan
        g = v.gt_types.astype(float)
        g[g == 3] = np.nan

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y)
        g2 = g[mask]
        y2 = y[mask]

        # Skip if not enough samples
        if len(g2) < 3:
            continue

        # MAF on non-missing genotypes
        maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
        if maf < 0.01:
            continue

        # Regression on filtered samples
        y0 = y2 - y2.mean()
        g0 = g2 - g2.mean()
        df = len(y2) - 2

        denom = g0 @ g0

        # Skip monomorphic SNPs (only 1 allele present in all samples)
        if denom == 0:
            continue

        beta = (g0 @ y0) / denom

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")
        count += 1